# 第 7 周 - 笔记本 2：基线模型测试

## 练习目标（理念）

在微调之前测试基座 **Llama 3.2**，并与第 6 周其它基线对比：

1. 以 **4-bit 量化** 加载基座模型
2. 在样例商品上试推理
3. 在测试集上做正式评估
4. 用柱状图建立「微调前」的绩效基线

**预期结果：** 大约 **~$110** 误差（明显差于简单常数基线）

## 怎么跑

1. 需要 GPU（本地或 Colab）
2. `.env` 提供 `HF_TOKEN`；配置指向正确数据集/模型
3. 预计 **15–20 分钟**（完整评估较慢）


In [ ]:
# ========== 导入与鉴权 ==========

# sys：改模块搜索路径
import sys
# 让 notebooks/ 能找到上级 src 包
sys.path.append('..')

# os：读环境变量；torch：CUDA / 张量
import os
import torch
# load_dotenv：从 .env 注入密钥
from dotenv import load_dotenv
# login：Hugging Face Hub 鉴权
from huggingface_hub import login
# 分词器、因果 LM、4-bit 量化配置
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 项目内：样本、评估器、配置
from src.items import Item
from src.evaluator import evaluate
from src.config import config

# 加载环境变量
load_dotenv()
# 取 HF token 并登录
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

# 确认环境与 GPU 可用性
print("✅ Environment loaded")
print(f"GPU available: {torch.cuda.is_available()}")


## 配置

核对 `config`：模型名、数据集、`EVAL_SIZE`、`MAX_TOKENS` 等是否符合预期。


In [ ]:
# 打印当前配置表，避免指错仓库或评估样本量
config.display()


## 加载测试数据

只加载 **test** 子集，用于基线推理与指标计算。


In [ ]:
# 标明数据来源
print(f"Loading test data from: {config.DATASET_NAME}")
# 只要 test；train/val 用占位丢弃
_, _, test = Item.from_hub(config.DATASET_NAME)

# 打印测试集规模
print(f"✅ Loaded {len(test):,} test items")


## 使用 4-bit 量化加载基座模型

BitsAndBytes 的 NF4 + double quant：在有限显存上跑 Llama，并为后续 QLoRA 路线对齐配置。


In [ ]:
# 配置 4-bit 量化（QLoRA 常用前置）
bnb_config = BitsAndBytesConfig(
    # 4-bit 加载权重
    load_in_4bit=True,
    # 量化格式 nf4
    bnb_4bit_quant_type="nf4",
    # 前向计算用 float16
    bnb_4bit_compute_dtype=torch.float16,
    # 双重量化进一步省显存
    bnb_4bit_use_double_quant=True,
)

# 提示加载目标与耗时
print(f"Loading base model: {config.BASE_MODEL}")
print("This may take a few minutes...")

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
# 加载量化后的因果语言模型；自动映射设备
model = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 补齐 pad_token（许多生成模型默认只有 eos）
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

# 打印成功与显存足迹
print("✅ Model loaded")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")


## 对样例商品试推理

定义 `predict_base`，先看 5 条样本的「真实价 vs 预测」，建立直觉。


In [ ]:
def predict_base(item: Item) -> str:
    """未微调基座模型的价格预测：构造/复用 prompt，generate 短补全，再按 PREFIX 截取。"""
    # 没有现成 prompt 就现场 make_prompts（测试集保留精确价格）
    if not item.prompt:
        item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=False)

    # 推理用输入文本（不含答案 completion）
    prompt = item.test_prompt()

    # 分词并放到模型 device
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 无梯度生成，省显存
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    # 解码完整序列
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 用 PREFIX 分割，取模型补全段
    completion = response.split(config.PREFIX)[-1].strip()

    return completion


In [ ]:
# 快速抽查 5 条，确认管线能跑通
print("Testing base model on 5 sample products:\n")

for i in range(5):
    item = test[i]
    # 基座预测
    prediction = predict_base(item)

    # 打印对比（文案保持原样）
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"Predicted: {prediction}")
    print("-" * 60)


## 对测试集全面评估

在 `EVAL_SIZE` 条上跑 `evaluate`，得到微调前的正式误差数字。


In [ ]:
# 提示评估规模与耗时预期
print(f"Evaluating on {config.EVAL_SIZE} test items...")
print("This will take 10-15 minutes...\n")

# workers=1：GPU 推理用单进程更稳
results = evaluate(
    predict_base,
    test,
    size=config.EVAL_SIZE,
    workers=1  # Sequential for GPU
)

# 打印「微调前」基线结果块
print(f"\n{'='*60}")
print("BASE MODEL RESULTS (BEFORE FINE-TUNING)")
print(f"{'='*60}")
print(f"Average Error: ${results['average_error']:.2f}")
print(f"MSE: {results['mse']:,.0f}")
print(f"R²: {results['r2']:.1f}%")
print(f"{'='*60}")


## 可视化结果

把本笔记本得到的 Base Llama 误差，叠到第 6 周各基线柱状图上，直观看「差在哪里」。


In [ ]:
# 导入 plotly：交互式柱状图
import plotly.graph_objects as go

# 与其它模型对比（数值来自第 6 周；最后一项用本次 results['average_error']）
baseline_results = [
    ("Constant", "gray", 106.18),
    ("Linear Regression", "gray", 101.56),
    ("NLP + LR", "gray", 76.81),
    ("Random Forest", "gray", 72.28),
    ("XGBoost", "gray", 68.23),
    ("Human (Ed)", "black", 87.62),
    ("Neural Network", "orange", 63.97),
    ("GPT 4.1 Nano", "slateblue", 62.51),
    ("Grok 4.1 Fast", "slateblue", 57.62),
    ("Gemini 3 Pro", "slateblue", 50.54),
    ("Claude 4.5 Sonnet", "slateblue", 47.10),
    ("GPT 5.1", "slateblue", 44.74),
    ("Deep Neural Network", "orange", 46.49),
    ("Base Llama 3.2 4-bit", "darkred", results['average_error']),
]

# 拆成标签、颜色、误差值三列
labels, colors, values = zip(*baseline_results)

# 柱状图：x=模型名，y=平均绝对误差
fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

# 布局：标题、坐标轴、尺寸保持原样
fig.update_layout(
    title="Base Llama 3.2 vs Other Models - Price Prediction Error",
    yaxis=dict(range=[0, max(values)], title="Mean Absolute Error ($)"),
    xaxis=dict(tickangle=-45),
    width=1200,
    height=600,
)

# 在笔记本中展示交互图
fig.show()

# 解读：基座往往比常数基线还差——这是预期，说明必须微调
print(f"\n⚠️ Base Llama performs poorly: ${results['average_error']:.2f} error")
print(f"   Worse than simple constant baseline!")
print(f"\n💡 This is expected - the model hasn't been trained for this task")
print(f"   Fine-tuning should reduce error to ~$40-65")


## 小结

✅ **基线已建立！**

**基座模型表现：**

- 误差大约 **$110**（非常差）
- 往往比简单常数基线还糟
- 模型尚未理解本任务

**为什么这么差？**

- Llama 3.2 没有针对价格预测做过训练
- 缺少产品定价领域知识
- 需要微调才能学会任务格式与规律

**微调会做什么：**

- 教会模型根据商品信息预测价格
- 从训练数据里学模式
- 预期误差下降约 60–70%
- 目标大约 **$40–65**（逼近 GPT-5.1 一带）

**下一步：** `03_qlora_training.ipynb` —— 在 Google Colab 上用 QLoRA 微调。
